# Part 3: The Logistics Commander (CoT & ToT) (20 Points)
- **The Problem**: You have limited resources and multiple victims.
- **Your Task**:
    - Load data/incidents.txt (3 critical incidents).
    - Situation: You have ONE rescue boat at Ragama.
    - Step A (CoT - Scoring): Use cot_reasoning.v1.
      - Prompt the model to analyse each incident row by row.
      - Logic: Assign a Priority Score (1-10) based on Age, Need, and Urgency.
          - Example Logic:
  
              Base Score: 5

              +2 if Age > 60 or < 5

              +3 if Need == ”Rescue” (Life Threat)

              +1 if Need == ”Insulin/Medicine”

              Result: ” Score: 8/10”
              
    - Step B (ToT - Strategy): Use tot_reasoning.v1.
      - Constraint: Travel times: Ragama → Ja-Ela (10m), Ja-Ela → Gampaha (40m).
      - Task: Explore 3 distinct branches:
      - Branch 1: Save the highest score first (Greedy).
      - Branch 2: Save closest first (speed).
      - Branch 3: Save furthest first (Logistics).
    - Goal: maximize total priority score saved within the shortest time (assume boat can handle one incident per stop, unlimited capacity for simplicity). incident per stop, unlimited capacity for simplicity).
    - Outcome: The model must select the optimal route.

In [1]:
# Import libraries
import sys
sys.path.append('..')

from utils.token_utils import pick_encoding, count_text_tokens
from utils.logging_utils import log_llm_call
from utils.prompts import render
from utils.llm_client import LLMClient
from utils.router import pick_model
import tiktoken
import pandas as pd
from IPython.display import Markdown, display

In [2]:
# pick reasoning model
reasoning_model = pick_model('groq', 'cot')
print(f'Using reasoning model: {reasoning_model}')

client_reasoning = LLMClient('groq', reasoning_model)

Using reasoning model: openai/gpt-oss-safeguard-20b


In [3]:
with open("../data/Incidents.txt", "r") as f:
    incidents_text = f.read()

print("Incidents loaded:")
print(incidents_text)

Incidents loaded:
ID | Time    | Area    | People | Ages   | Main Need | Message
1  | 08:00 AM| Gampaha | 4      | 20-40  | Water     | "Thirsty but safe on roof. Water level stable." 
2  | 08:15 AM| Ja-Ela  | 1      | 75     | Insulin   | "Diabetic, missed dose yesterday. Feeling faint."
3  | 08:20 AM| Ragama  | 2      | 10, 35 | Rescue    | "Water approaching neck level. Child is crying."


## Step A

In [14]:
problem =''
instructions = f'''
Solve the following problem step by step.
1. Analyze the incidents and identify Age, Need, and Urgency for each incident.
2. Assign a priority score from 1 to 10 based on the identified factors, where 10 is the highest priority.
3. Base Score of the incident is 5.
4. +2 if Age > 60 or < 5
5. +3 if Need == ”Rescue” (Life Threat)
6. +1 if Need == ”Insulin/Medicine”

Analyze each incident and provide your reasoning for the assigned priority score. Clearly mention the thought process and then show formatted output as below:
"Location | Age | Need | Urgency | Score | Reasoning"

Incidents:\n
{incidents_text}
'''


prompt_text, spec = render(
    'cot_reasoning.v1',
    role='Crisis logistic analyst',
    problem=problem
)

# Combine problem with instruction
full_prompt = f"""text: {prompt_text}

instruction: {instructions}"""

display(Markdown(full_prompt))

text: You are Crisis logistic analyst. Solve the problem carefully.
Problem: 

First, outline your reasoning steps briefly.
Then provide the final answer clearly marked under 'Answer:'.
Keep reasoning concise; avoid unnecessary prose.


instruction: 
Solve the following problem step by step.
1. Analyze the incidents and identify Age, Need, and Urgency for each incident.
2. Assign a priority score from 1 to 10 based on the identified factors, where 10 is the highest priority.
3. Base Score of the incident is 5.
4. +2 if Age > 60 or < 5
5. +3 if Need == ”Rescue” (Life Threat)
6. +1 if Need == ”Insulin/Medicine”

Analyze each incident and provide your reasoning for the assigned priority score. Clearly mention the thought process and then show formatted output as below:
"Location | Age | Need | Urgency | Score | Reasoning"

Incidents:

ID | Time    | Area    | People | Ages   | Main Need | Message
1  | 08:00 AM| Gampaha | 4      | 20-40  | Water     | "Thirsty but safe on roof. Water level stable." 
2  | 08:15 AM| Ja-Ela  | 1      | 75     | Insulin   | "Diabetic, missed dose yesterday. Feeling faint."
3  | 08:20 AM| Ragama  | 2      | 10, 35 | Rescue    | "Water approaching neck level. Child is crying."


In [15]:
messages = [{'role': 'user', 'content': full_prompt}]
response = client_reasoning.chat(messages, temperature=spec.temperature, max_tokens=spec.max_tokens)
display(Markdown(response['text']))
log_llm_call('groq', reasoning_model, 'cot', response['latency_ms'], response['usage'])

**Answer:**

| Location | Age | Need | Urgency | Score | Reasoning |
|----------|-----|------|---------|-------|-----------|
| Gampaha | 20‑40 | Water | Low – “Thirsty but safe on roof. Water level stable.” | 5 | Base 5. No age bonus (20‑40). Need is not Rescue or Insulin, so no extra points. |
| Ja‑Ela | 75 | Insulin | Moderate – “Diabetic, missed dose yesterday. Feeling faint.” | 8 | Base 5 +2 (age >60) +1 (Insulin/Medicine) = 8. |
| Ragama | 10 & 35 | Rescue | High – “Water approaching neck level. Child is crying.” | 8 | Base 5 +3 (Rescue) = 8. No age bonus (10 is >5). |

**Reasoning Summary**

1. **Incident 1**: Ages 20‑40 → no age bonus. Need is “Water,” not a life‑threat or medication need → no extra points. Score stays at the base 5. Urgency is low because the water level is stable and people are on a roof.  
2. **Incident 2**: Age 75 → qualifies for +2. Need is “Insulin” → +1. Total 5+2+1 = 8. Urgency is moderate due to faintness from missed dose.  
3. **Incident 3**: Ages 10 and 35 → none qualify for age bonus. Need is “Rescue” → +3. Total 5+3 = 8. Urgency is high because water is at neck level and a child is crying.

In [21]:
priority_scores = response['text'].strip().split('\n\n')[1]
display(Markdown("### Priority Scores\n" + priority_scores))

### Priority Scores
| Location | Age | Need | Urgency | Score | Reasoning |
|----------|-----|------|---------|-------|-----------|
| Gampaha | 20‑40 | Water | Low – “Thirsty but safe on roof. Water level stable.” | 5 | Base 5. No age bonus (20‑40). Need is not Rescue or Insulin, so no extra points. |
| Ja‑Ela | 75 | Insulin | Moderate – “Diabetic, missed dose yesterday. Feeling faint.” | 8 | Base 5 +2 (age >60) +1 (Insulin/Medicine) = 8. |
| Ragama | 10 & 35 | Rescue | High – “Water approaching neck level. Child is crying.” | 8 | Base 5 +3 (Rescue) = 8. No age bonus (10 is >5). |

## Step B

In [54]:
tot_problem = f'''
Incidents (in Urgency column) and their Priority Scores:
{priority_scores}

Travel times:
- Ragama → Ja-Ela (10m)
- Ja-Ela → Gampaha (40m)

Constraints:
- have ONE rescue boat at Ragama
- boat can handle one incident per stop
- unlimited capacity for simplicity

Branches:
- 1: Save the highest score first (Greedy).
- 2: Save closest first (speed).
- 3: Save furthest first (Logistics).

Goal: maximize total priority score saved within the shortest time.

Output Format: 
- Summary table:
    Branch | Rescue Order (Locations) | Total Score Saved | Total Time Taken | Reasoning 
- Optimal Branch: [Optimal branch and the justification for why it's optimal]
- Hypothesis → Steps → Intermediate check for all branches
'''

prompt_text_tot, spec_tot = render(
    'tot_reasoning.v1',
    role='Crisis logistics commander',
    problem=tot_problem,
    branches='3'
)

display(Markdown(prompt_text_tot))

You are Crisis logistics commander. Explore 3 distinct solution paths to the problem below.
Problem: 
Incidents (in Urgency column) and their Priority Scores:
| Location | Age | Need | Urgency | Score | Reasoning |
|----------|-----|------|---------|-------|-----------|
| Gampaha | 20‑40 | Water | Low – “Thirsty but safe on roof. Water level stable.” | 5 | Base 5. No age bonus (20‑40). Need is not Rescue or Insulin, so no extra points. |
| Ja‑Ela | 75 | Insulin | Moderate – “Diabetic, missed dose yesterday. Feeling faint.” | 8 | Base 5 +2 (age >60) +1 (Insulin/Medicine) = 8. |
| Ragama | 10 & 35 | Rescue | High – “Water approaching neck level. Child is crying.” | 8 | Base 5 +3 (Rescue) = 8. No age bonus (10 is >5). |

Travel times:
- Ragama → Ja-Ela (10m)
- Ja-Ela → Gampaha (40m)

Constraints:
- have ONE rescue boat at Ragama
- boat can handle one incident per stop
- unlimited capacity for simplicity

Branches:
- 1: Save the highest score first (Greedy).
- 2: Save closest first (speed).
- 3: Save furthest first (Logistics).

Goal: maximize total priority score saved within the shortest time.

Output Format: 
- Summary table:
    Branch | Rescue Order (Locations) | Total Score Saved | Total Time Taken | Reasoning 
- Optimal Branch: [Optimal branch and the justification for why it's optimal]
- Hypothesis → Steps → Intermediate check for all branches


For each path, provide: Hypothesis → Steps → Intermediate check.
After exploring, select the best path and provide the final 'Answer:'.


In [55]:
response_tot = client_reasoning.chat([{'role': 'user', 'content': prompt_text_tot}], temperature=spec_tot.temperature, max_tokens=spec_tot.max_tokens)
display(Markdown(response_tot['text']))
log_llm_call('groq', reasoning_model, 'tot', response_tot['latency_ms'], response_tot['usage'])


**Crisis‑Logistics Decision Matrix**

| Branch | Rescue Order (Locations) | Total Score Saved | Total Time Taken | Reasoning |
|--------|--------------------------|-------------------|------------------|-----------|
| 1 – Greedy | **Ragama → Ja‑Ela → Gampaha** | 21 | 50 min | Highest‑score incidents (8 + 8) handled first; starting at Ragama saves the 10‑m leg to Ja‑Ela, then the 40‑m leg to Gampaha. |
| 2 – Closest | **Ragama → Ja‑Ela → Gampaha** | 21 | 50 min | The only “closest‑first” path that is feasible from Ragama is the same as the greedy path (10 m to Ja‑Ela, then 40 m to Gampaha). |
| 3 – Furthest | **Ragama → Gampaha → Ja‑Ela** | 21 | 90 min | To reach the furthest point first you must travel 50 m to Gampaha (via Ja‑Ela), then 40 m back to Ja‑Ela.  This doubles the travel time with no score benefit. |

---

### Optimal Branch

**Branch 1 (Greedy)** is optimal.  
It achieves the maximum possible score (21) while keeping the travel time minimal (50 min). Branch 2 yields identical performance, but Branch 1 has the advantage of prioritising the two highest‑score incidents first, which is a more defensible operational policy.

---

## Branch‑by‑Branch Walk‑through

### Branch 1 – Greedy

**Hypothesis**  
Rescuing the highest‑score incidents first (Ja‑Ela & Ragama) will maximise the total score while still staying within a short travel window.

**Steps**  
1. **Start at Ragama** – rescue the 8‑point incident (child in water).  
2. **Travel Ragama → Ja‑Ela** (10 min).  
3. **Rescue Ja‑Ela** – 8‑point insulin case.  
4. **Travel Ja‑Ela → Gampaha** (40 min).  
5. **Rescue Gampaha** – 5‑point water‑access case.

**Intermediate Check**  
- Total travel = 10 + 40 = 50 min.  
- All three incidents rescued, total score = 8 + 8 + 5 = 21.

### Branch 2 – Closest

**Hypothesis**  
Prioritising the nearest incident first will minimise travel time; the resulting order will still capture all high‑score incidents.

**Steps**  
1. **Start at Ragama** – rescue 8‑point incident (child).  
2. **Travel Ragama → Ja‑Ela** (10 min).  
3. **Rescue Ja‑Ela** – 8‑point insulin case.  
4. **Travel Ja‑Ela → Gampaha** (40 min).  
5. **Rescue Gampaha** – 5‑point water case.

**Intermediate Check**  
- Travel time = 10 + 40 = 50 min (identical to Branch 1).  
- Total score = 21.

### Branch 3 – Furthest

**Hypothesis**  
Going to the furthest location first might reduce the number of back‑tracks, but with only three sites the benefit is negligible.

**Steps**  
1. **Start at Ragama** – rescue 8‑point incident (child).  
2. **Travel Ragama → Ja‑Ela → Gampaha** (10 + 40 = 50 min).  
3. **Rescue Gampaha** – 5‑point water case.  
4. **Travel Gampaha → Ja‑Ela** (40 min).  
5. **Rescue Ja‑Ela** – 8‑point insulin case.

**Intermediate Check**  
- Total travel = 50 + 40 = 90 min.  
- Total score = 21.

---

**Answer:** Branch 1 (Greedy) is the optimal strategy – it delivers the maximum priority score (21) in the shortest time (50 min). Branch 2 is a viable alternative with identical metrics, while Branch 3 is sub‑optimal due to excessive travel.